# 01 — Setup Tables

Load `./app/data/source/orders.csv` into **four table formats** side-by-side:

| # | Format | Catalog / Location |
|---|--------|--------------------|
| 1 | **Iceberg** | `iceberg_catalog.my_database.orders_iceberg` |
| 2 | **Delta** | `.tmp/local_delta_warehouse/my_database/orders_delta` |
| 3 | **Hudi** | `.tmp/local_hudi_warehouse/my_database/orders_hudi` |
| 4 | **Parquet** | `.tmp/local_parquet_warehouse/my_database/orders_parquet` |

> **Pre-requisite:** `spark-defaults.conf` must include Iceberg, Delta, and Hudi packages + extensions.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Spark-Developer").master("local[*]").getOrCreate()

spark

:: loading settings :: url = jar:file:/usr/local/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/saketkumar/.ivy2.5.2/cache
The jars for the packages stored in: /Users/saketkumar/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
io.delta#delta-spark_2.13 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b3e0085e-c0c5-4380-a8ca-1945bac3a3c0;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in central
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.2 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4

---
## Step 1 — Read CSV with explicit schema

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("order_id",      IntegerType(), nullable=False),
    StructField("customer_name", StringType(),  nullable=False),
    StructField("product",       StringType(),  nullable=False),
    StructField("quantity",      IntegerType(), nullable=False),
    StructField("unit_price",    DoubleType(),  nullable=False),
    StructField("order_date",    StringType(),  nullable=False),
    StructField("status",        StringType(),  nullable=False),
])

orders_df = spark.read.csv("./app/data/source/orders.csv", header=True, schema=schema)
orders_df.printSchema()
orders_df.show(truncate=False)

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pendin

---
## 1 — Iceberg

- Open table format with **ACID transactions** and **time travel**
- Catalog: `iceberg_catalog` (Hadoop catalog, configured in `spark-defaults.conf`)
- Unique feature: **snapshot-based versioning**

In [3]:
orders_df.writeTo("iceberg_catalog.my_database.orders_iceberg") \
    .tableProperty("format-version", "2") \
    .createOrReplace()

print("Written to Iceberg: iceberg_catalog.my_database.orders_iceberg")

Written to Iceberg: iceberg_catalog.my_database.orders_iceberg


In [4]:
iceberg_df = spark.table("iceberg_catalog.my_database.orders_iceberg")
iceberg_df.show(truncate=False)
print(f"Rows: {iceberg_df.count()}")

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pending  |
|7       |Grace Kim    |USB Hub     |1       |49.99     |2024-01-21|completed|
|8       |Henry Wilson |Laptop Stand|2       |59.99     |2024-01-22|shipped  |
|9       |Iris Chen    |SSD Drive   |1       |119.99    |2024-01-23|completed|
|10      |James Davis  |Microphone  |1       |199.99

In [5]:
# Snapshot history (time travel feature)
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM iceberg_catalog.my_database.orders_iceberg.snapshots
""").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|1382764218072417236|2026-04-25 12:42:21.885|overwrite|
|8790512653395721002|2026-04-25 12:42:29.154|overwrite|
|1317946826273725722|2026-04-25 18:34:41.501|overwrite|
+-------------------+-----------------------+---------+



---
## 2 — Delta

- Delta Lake with **ACID transactions** and **schema enforcement**
- Path-based write; reads via `spark.read.format("delta")`
- Unique feature: **transaction log** stored in `_delta_log/`

In [6]:
DELTA_PATH = ".tmp/local_delta_warehouse/my_database/orders_delta"

orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(DELTA_PATH)

print(f"Written to Delta: {DELTA_PATH}")

26/04/26 00:04:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Written to Delta: .tmp/local_delta_warehouse/my_database/orders_delta


In [7]:
delta_df = spark.read.format("delta").load(DELTA_PATH)
delta_df.show(truncate=False)
print(f"Rows: {delta_df.count()}")

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pending  |
|7       |Grace Kim    |USB Hub     |1       |49.99     |2024-01-21|completed|
|8       |Henry Wilson |Laptop Stand|2       |59.99     |2024-01-22|shipped  |
|9       |Iris Chen    |SSD Drive   |1       |119.99    |2024-01-23|completed|
|10      |James Davis  |Microphone  |1       |199.99

In [8]:
# Transaction log history
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, DELTA_PATH)
dt.history().select("version", "timestamp", "operation").show(truncate=False)

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|2      |2026-04-25 18:34:44.922|WRITE    |
|1      |2026-04-25 12:43:36.36 |WRITE    |
|0      |2026-04-25 12:42:33.818|WRITE    |
+-------+-----------------------+---------+



---
## 3 — Hudi

- Apache Hudi with **upserts** and **incremental processing**
- Path-based table (no catalog required)
- Unique feature: **record-level upserts** via `recordkey.field`

> **Setup:** Add to `spark-defaults.conf`:
> ```
> spark.jars.packages   ...,org.apache.hudi:hudi-spark3.5-bundle_2.13:1.0.2
> spark.sql.extensions  ...,org.apache.spark.sql.hudi.HoodieSparkSessionExtension
> ```
> Check [Hudi releases](https://hudi.apache.org/releases/) for the bundle matching your Spark version.

In [9]:
# HUDI_PATH = ".tmp/local_hudi_warehouse/my_database/orders_hudi"

# hudi_options = {
#     "hoodie.table.name":                       "orders_hudi",
#     "hoodie.datasource.write.recordkey.field":  "order_id",
#     "hoodie.datasource.write.precombine.field": "order_date",
#     "hoodie.datasource.write.operation":        "bulk_insert",
#     "hoodie.datasource.write.table.type":       "COPY_ON_WRITE",
# }

# orders_df.write \
#     .format("hudi") \
#     .options(**hudi_options) \
#     .mode("overwrite") \
#     .save(HUDI_PATH)

# print(f"Written to Hudi: {HUDI_PATH}")

In [10]:
# # Hudi adds internal metadata columns — select only the original ones
# original_cols = ["order_id", "customer_name", "product", "quantity", "unit_price", "order_date", "status"]

# hudi_df = spark.read.format("hudi").load(HUDI_PATH)
# hudi_df.select(original_cols).show(truncate=False)
# print(f"Rows: {hudi_df.count()}")

---
## 4 — Parquet

- Raw columnar file format — **no catalog, no ACID, no versioning**
- Fastest reads; baseline to compare against managed table formats
- Unique feature: **column pruning** and **predicate pushdown** at file level

In [11]:
PARQUET_PATH = ".tmp/local_parquet_warehouse/my_database/orders_parquet"

orders_df.write \
    .mode("overwrite") \
    .parquet(PARQUET_PATH)

print(f"Written to Parquet: {PARQUET_PATH}")

Written to Parquet: .tmp/local_parquet_warehouse/my_database/orders_parquet


In [12]:
parquet_df = spark.read.parquet(PARQUET_PATH)
parquet_df.show(truncate=False)
print(f"Rows: {parquet_df.count()}")

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pending  |
|7       |Grace Kim    |USB Hub     |1       |49.99     |2024-01-21|completed|
|8       |Henry Wilson |Laptop Stand|2       |59.99     |2024-01-22|shipped  |
|9       |Iris Chen    |SSD Drive   |1       |119.99    |2024-01-23|completed|
|10      |James Davis  |Microphone  |1       |199.99

---
## Summary

| Format | ACID | Time Travel / History | Schema Evolution | Upserts | Catalog |
|--------|------|----------------------|------------------|---------|--------|
| **Iceberg** | Yes | Snapshots | Yes | Yes (MOR) | `iceberg_catalog` |
| **Delta** | Yes | Transaction log | Yes | Yes | `spark_catalog` |
| **Hudi** | Yes | Commit timeline | Yes | Yes (COW/MOR) | Path-based |
| **Parquet** | No | No | No | No | Path-based |